# CRISPRa sgRNA re-annotation — Phase 3-5: genomic alignment, PAM/gene annotation, twin revisit

Takes the reconciled `CRISPRa_targeting_sgRNA_master` table from `dedup_frameshift_twins.ipynb` and adds
the **locus-authoritative** layer that the sequence-only steps could not:

- **Phase 3** — align each guide's 19 bp discriminating window to hg38 with bowtie2 (perfect-match,
  all loci), split into uniquely-aligned / multi-mapping / unaligned.
- **Phase 4** — NGG-PAM check downstream of each protospacer; resolve multi-mappers by PAM + gene proximity.
- **Phase 5** — assign the **current gene at each guide's genomic locus** (nearest TSS): confirm on-target
  for resolved names, and **settle the 208 symbols HGNC left ambiguous/unresolved** by their locus
  (e.g. `AKAP2 → PALM2AKAP2`). Compare against the designed target → final gene call + off-target flags.
- **Frameshift-twin revisit** — (a) confirm every sequence-based twin maps to genomically adjacent
  (±1 bp) loci; (b) run twin detection *directly on alignment* (the 19 bp genomic unit) to catch pairs the
  20 bp-protospacer sequence method missed; (c) finalize the twin/dedup set using locus + current gene.

**Environment:** run under `gwt-env` (`/Users/rzhu/miniconda3/envs/gwt-env`) which has bowtie2, pysam,
biopython, pyfaidx, gffutils. Genome files are referenced by absolute path from the sibling GRNPerturbSeq
project. Local `genome/genes_df_subset_for_sgRNA_annotation.parquet` holds the CRISPRa-scoped gene models.

In [1]:
import os
import sys
import shutil
import subprocess
from collections import defaultdict

import numpy as np
import pandas as pd
import pysam
from pyfaidx import Fasta

from sgRNAalign_util import (
    build_tss_index, nearest_tss, distance_to_gene_tss, has_ngg_pam, genes_near,
    frameshift_twin_pairs, collapse_twin_groups,
)

# --- paths ---
RESULTS = 'results'
GENOME = 'genome'
HG38_DIR = '/Users/rzhu/Gladstone Dropbox/Ronghui Zhu/GRNPerturbSeq/2_files/hg38_genome'
BT2_INDEX = f'{HG38_DIR}/hg38'          # bowtie2 index prefix
HG38_FA = f'{HG38_DIR}/hg38.fa'         # for PAM lookup
# bowtie2 lives in the same conda-env bin as the Python running this notebook (gwt-env);
# fall back to PATH. This makes headless `jupyter nbconvert --execute` work too.
BOWTIE2 = os.path.join(os.path.dirname(sys.executable), 'bowtie2')
if not os.path.exists(BOWTIE2):
    BOWTIE2 = shutil.which('bowtie2') or 'bowtie2'

TSS_WINDOW = 2000   # bp: a guide is "near" a gene's TSS within this distance (CRISPRa activation window)
os.makedirs(RESULTS, exist_ok=True)

master = pd.read_parquet(f'{RESULTS}/CRISPRa_targeting_sgRNA_master.parquet')
# Preserve the Phase-2 (sequence-based) twin columns under a seq_ prefix so the Phase 3-5
# alignment-based twin recompute below doesn't collide with them on merge.
master = master.rename(columns={
    'twin_group_id': 'seq_twin_group_id', 'is_representative': 'seq_twin_representative',
    'collapsed_into': 'seq_twin_collapsed_into', 'is_twin': 'seq_is_twin', 'keep': 'seq_keep'})
genes = pd.read_parquet(f'{GENOME}/genes_df_subset_for_sgRNA_annotation.parquet')
print(f'master guides : {len(master):,}   genes_df_subset : {len(genes):,}   bowtie2: {BOWTIE2}')

master guides : 77,412   genes_df_subset : 24,047   bowtie2: /Users/rzhu/miniconda3/envs/gwt-env/bin/bowtie2


## Phase 3 — Align the 19 bp discriminating window to hg38

Export `protospacer[1:]` (last 19 bp) per guide, then bowtie2 end-to-end with a **full-length exact seed**
(`-L 19 -N 0`) and `--score-min C,0,0` so only **perfect** genomic matches are reported, with `-a` to list
**all** exact loci (true paralog multi-mapping). Guides that align 0 times have no perfect hg38 match — the
"doesn't align to current assembly" signal.

In [2]:
# Export 19 bp FASTA (last 19 of the 20 bp protospacer)
fasta_path = f'{RESULTS}/CRISPRa_targeting_sgRNA.fa'
with open(fasta_path, 'w') as f:
    for gid, proto in zip(master['guide_id'], master['protospacer']):
        f.write(f'>{gid}\n{proto[1:]}\n')
print(f'wrote {len(master):,} records -> {fasta_path}')

# Run bowtie2 (skip if the SAM already exists; delete it to force re-run)
sam_path = f'{RESULTS}/CRISPRa_targeting_sgRNA_alignment.sam'
if not os.path.exists(sam_path):
    cmd = [BOWTIE2, '-x', BT2_INDEX, '-f', fasta_path, '-S', sam_path,
           '--end-to-end', '-L', '19', '-N', '0', '--score-min', 'C,0,0',
           '-a', '-p', '8']
    print('running:', ' '.join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stderr.strip().splitlines()[-6:])
else:
    print(f'using existing {sam_path}')

wrote 77,412 records -> results/CRISPRa_targeting_sgRNA.fa
using existing results/CRISPRa_targeting_sgRNA_alignment.sam


In [3]:
# Parse SAM: per-guide list of aligned loci (chrom, pos, strand). Also count loci.
aln = defaultdict(list)
with pysam.AlignmentFile(sam_path, 'r') as sam:
    for r in sam:
        if r.is_unmapped:
            continue
        aln[r.query_name].append((r.reference_name, int(r.reference_start),
                                  '-' if r.is_reverse else '+'))

n_loci = master['guide_id'].map(lambda g: len(aln.get(g, [])))
master['n_loci'] = n_loci.values

unaligned = master[master['n_loci'] == 0]
unique = master[master['n_loci'] == 1]
multi = master[master['n_loci'] > 1]
print(f'unaligned (0 loci) : {len(unaligned):,}')
print(f'unique   (1 locus) : {len(unique):,}')
print(f'multi   (>1 loci)  : {len(multi):,}')
print('multi-locus count distribution (capped view):',
      master.loc[master.n_loci > 1, 'n_loci'].clip(upper=10).value_counts().sort_index().to_dict())
unaligned[['guide_id', 'target_gene_symbol', 'current_symbol', 'protospacer']]

unaligned (0 loci) : 2
unique   (1 locus) : 71,008
multi   (>1 loci)  : 6,402
multi-locus count distribution (capped view): {2: 3975, 3: 945, 4: 348, 5: 274, 6: 156, 7: 216, 8: 159, 9: 28, 10: 301}


,guide_id,target_gene_symbol,current_symbol,protospacer
47726,DGAT1-3,DGAT1,DGAT1,GCACCGGCGCAGAACACCCT
52344,GPR3-4,GPR3,GPR3,GAGCGCGGGCAGACTCGGGA


## Phase 4-5 — PAM check, multi-locus resolution, and locus-based gene assignment

For every guide we pick a **primary locus** and assign the **current gene** there:

1. If the reconciled designed gene's TSS is within `TSS_WINDOW` of any of the guide's loci → pick that
   locus (`designed_confirmed`, on-target).
2. Otherwise pick the locus nearest to *any* gene's TSS and take that gene as the call
   (`locus_reassigned` when a designed gene was known but its TSS isn't near — a target mismatch; or
   `locus_assigned` when the designed name was unresolved — the locus settles it, e.g. `AKAP2→PALM2AKAP2`).
3. NGG-PAM is checked immediately 3' of the chosen locus.

`nearest_tss` uses a per-chromosome sorted-TSS binary search, so all ~626k alignment records resolve in
seconds.

In [4]:
# Setup: TSS index, per-gene TSS lookup, genome handle
tss_index = build_tss_index(genes)
tss_by_gene = dict(zip(genes['gene_id'], genes['tss']))
genome = Fasta(HG38_FA)
designed_gid = dict(zip(master['guide_id'], master['gene_id']))

INF = float('inf')
def _d(x):
    """NaN-safe distance -> +inf so it never wins a min()."""
    return x if (x is not None and x == x) else INF


def resolve_guide(guide):
    loci = aln.get(guide, [])
    if not loci:
        return dict(guide_id=guide, n_loci=0, chrom=np.nan, pos=np.nan, strand=np.nan,
                    has_pam=np.nan, locus_gene_id=np.nan, locus_gene_name=np.nan,
                    dist_locus_tss=np.nan, dist_designed_tss=np.nan,
                    assignment='unaligned')
    dgid = designed_gid.get(guide)
    dtss = tss_by_gene.get(dgid) if isinstance(dgid, str) else None

    best_designed = None   # (dist, chrom, pos, strand, lg_id, lg_name, lg_dist)
    best_locus = None      # (locus_tss_dist, chrom, pos, strand, lg_id, lg_name, lg_dist)
    for ch, p, s in loci:
        lg_id, lg_name, lg_d = nearest_tss(tss_index, ch, p)
        if best_locus is None or _d(lg_d) < _d(best_locus[0]):
            best_locus = (lg_d, ch, p, s, lg_id, lg_name, lg_d)
        if dtss is not None:
            dd = distance_to_gene_tss(p, dtss)
            if best_designed is None or _d(dd) < _d(best_designed[0]):
                best_designed = (dd, ch, p, s, lg_id, lg_name, lg_d)

    if best_designed is not None and _d(best_designed[0]) <= TSS_WINDOW:
        dd, ch, p, s, lg_id, lg_name, lg_d = best_designed
        assignment = 'designed_confirmed'
    else:
        _, ch, p, s, lg_id, lg_name, lg_d = best_locus
        dd = best_designed[0] if best_designed is not None else np.nan
        assignment = ('locus_reassigned' if isinstance(dgid, str) else 'locus_assigned')

    pam = has_ngg_pam(genome, ch, p, s)
    return dict(guide_id=guide, n_loci=len(loci), chrom=ch, pos=p, strand=s,
                has_pam=pam, locus_gene_id=lg_id, locus_gene_name=lg_name,
                dist_locus_tss=lg_d, dist_designed_tss=dd, assignment=assignment)


asg = pd.DataFrame([resolve_guide(g) for g in master['guide_id']])
print('assignment breakdown:')
print(asg['assignment'].value_counts().to_string())
print(f'\nhas NGG PAM (aligned guides): {asg[asg.n_loci>0].has_pam.mean():.1%}')

assignment breakdown:
assignment
designed_confirmed    75328
locus_reassigned       1282
locus_assigned          800
unaligned                 2

has NGG PAM (aligned guides): 99.9%


### Final gene call, name correction, and off-target flags

Merge the locus assignment onto the master table and produce the **final gene call** per guide:
`designed_confirmed` keeps the reconciled designed gene; `locus_reassigned`/`locus_assigned` take the gene
at the guide's locus. Flag guides that don't align, lack a PAM, target a mismatched gene, or are far from
any TSS.

In [5]:
annot = master.merge(asg, on='guide_id', how='left', suffixes=('', '_asg'))

# final gene call: designed_confirmed -> reconciled designed gene; else the locus gene
use_designed = annot['assignment'] == 'designed_confirmed'
annot['final_gene_id'] = np.where(use_designed, annot['gene_id'], annot['locus_gene_id'])
annot['final_gene_name'] = np.where(use_designed, annot['current_symbol'], annot['locus_gene_name'])

# (i) 'locus_reassigned' where the locus gene IS the designed gene (same id) but >TSS_WINDOW away:
#     on-target, just distal -- not a true mismatch.
designed_distal = (annot['assignment'] == 'locus_reassigned') & (annot['final_gene_id'] == annot['gene_id'])
annot.loc[designed_distal, 'assignment'] = 'designed_distal'

# (ii) alt-haplotype copies: locus gene has the SAME NAME as the designed gene but a different Ensembl
#      id (e.g. LILRA3 annotated separately on several *_alt scaffolds) -> still on-target for it.
same_name = ((annot['assignment'] == 'locus_reassigned') & annot['current_symbol'].notna()
             & (annot['final_gene_name'] == annot['current_symbol']))
annot.loc[same_name & (annot['dist_locus_tss'] <= TSS_WINDOW), 'assignment'] = 'designed_confirmed'
annot.loc[same_name & (annot['dist_locus_tss'] > TSS_WINDOW), 'assignment'] = 'designed_distal'

# (iii) retired / withdrawn identifiers that align only to an alt contig and resolve to no current gene
#       (e.g. LOC391322): keep the original library label as the target, leave Ensembl id empty.
alt_contig = annot['chrom'].astype(str).str.contains('_alt|_hap|_fix|_patch|_random', na=False)
retired = annot['name_resolution'].eq('unresolved') & annot['gene_id'].isna() & alt_contig
annot.loc[retired, 'final_gene_id'] = np.nan
annot.loc[retired, 'final_gene_name'] = annot.loc[retired, 'target_gene_symbol']
annot.loc[retired, 'assignment'] = 'retired_identifier'

# (iv) Pseudoautosomal-region (PAR) genes: a guide whose EVERY locus is on chrX/chrY and resolves to the
#      SAME gene is single-target -- the gene is identical on X and Y, so 2 (or more) loci is expected,
#      not off-target multi-mapping. Collapse to that PAR gene (representative id = the chrX copy).
def par_status(guide):
    loci = aln.get(guide, [])
    if len(loci) < 2:
        return (False, np.nan, np.nan)
    names, xid, anyid = set(), None, None
    for ch, p, s in loci:
        if ch not in ('chrX', 'chrY'):
            return (False, np.nan, np.nan)
        gid, gname, gd = nearest_tss(tss_index, ch, p)
        if not isinstance(gname, str):
            return (False, np.nan, np.nan)
        names.add(gname); anyid = gid
        if ch == 'chrX':
            xid = gid
    if len(names) == 1:
        return (True, next(iter(names)), xid if xid is not None else anyid)
    return (False, np.nan, np.nan)

par = [par_status(g) for g in annot['guide_id']]
annot['flag_par'] = [x[0] for x in par]
annot['_par_name'] = [x[1] for x in par]
annot['_par_id'] = [x[2] for x in par]
pm = annot['flag_par']
annot.loc[pm, 'final_gene_name'] = annot.loc[pm, '_par_name']
annot.loc[pm, 'final_gene_id'] = annot.loc[pm, '_par_id']
annot.loc[pm, 'assignment'] = 'par_gene'
annot.drop(columns=['_par_name', '_par_id'], inplace=True)

annot['final_gene_source'] = annot['assignment']

# flags (computed after all reclassifications)
annot['flag_unaligned'] = annot['assignment'] == 'unaligned'
annot['flag_no_pam'] = annot['has_pam'] == False
annot['flag_far_from_tss'] = annot['dist_locus_tss'] > TSS_WINDOW
annot['flag_target_mismatch'] = annot['assignment'] == 'locus_reassigned'      # true different-gene reassignment
annot['flag_designed_distal'] = annot['assignment'] == 'designed_distal'       # same gene, TSS >window away
annot['flag_name_rescued_by_locus'] = (annot['assignment'] == 'locus_assigned') & annot['final_gene_id'].notna()
annot['flag_retired_identifier'] = annot['assignment'] == 'retired_identifier'
# on-target = designed gene confirmed/distal, OR a PAR gene matching the designed gene
annot['on_target'] = (annot['assignment'].isin(['designed_confirmed', 'designed_distal'])
                      | (annot['flag_par'] & (annot['final_gene_name'] == annot['current_symbol'])))

print('final gene source:')
print(annot['final_gene_source'].value_counts().to_string())
print(f"\non-target for designed gene: {annot.on_target.mean():.1%}")
print(f"PAR genes (chrX/Y, single-target): {int(annot.flag_par.sum()):,}"
      f"  ({int((annot.flag_par & annot.on_target).sum())} on-target for designed gene)")
print(f"true different-gene reassignment  : {int(annot.flag_target_mismatch.sum()):,}")
print(f"retired identifiers               : {int(annot.flag_retired_identifier.sum()):,}")
annot.loc[annot.flag_par, ['guide_id', 'target_gene_symbol', 'final_gene_name', 'final_gene_id',
                           'n_loci', 'assignment', 'on_target']].head(8)

final gene source:
final_gene_source
designed_confirmed    75362
designed_distal         760
locus_assigned          713
locus_reassigned        481
par_gene                 63
retired_identifier       31
unaligned                 2

on-target for designed gene: 98.4%
PAR genes (chrX/Y, single-target): 63  (59 on-target for designed gene)
true different-gene reassignment  : 481
retired identifiers               : 31


,guide_id,target_gene_symbol,final_gene_name,final_gene_id,n_loci,assignment,on_target
1009,AKAP17A-1,AKAP17A,AKAP17A,ENSG00000197976,2,par_gene,True
1010,AKAP17A-2,AKAP17A,AKAP17A,ENSG00000197976,2,par_gene,True
2296,ASMT-1,ASMT,ASMT,ENSG00000196433,2,par_gene,True
2297,ASMT-2,ASMT,ASMT,ENSG00000196433,2,par_gene,True
2298,ASMTL-1,ASMTL,ASMTL,ENSG00000169093,2,par_gene,True
2299,ASMTL-2,ASMTL,ASMTL,ENSG00000169093,2,par_gene,True
5819,CD99-1,CD99,CD99,ENSG00000002586,2,par_gene,True
5820,CD99-2,CD99,CD99,ENSG00000002586,2,par_gene,True


## Frameshift-twin revisit (alignment-authoritative)

The Phase 2 twins were detected from the 20 bp protospacer sequence. Now we check them against the genome
and look for any the sequence method missed:

- **(a) Confirm** every sequence-based twin pair maps to genomically adjacent loci (same chrom/strand, |Δpos|
  = 1).
- **(b) Discover** twins directly from alignment: any two guides whose 19 bp windows map to offset-1 loci.
  This catches pairs where the twin relationship depends on the dropped 5' base (which the 20 bp-sequence
  method can miss), e.g. `AGO4-3/AGO4-4`.
- **(c) Finalize**: collapse twins that share the same **final (locus-based) gene**; keep offset-1 pairs
  whose two guides resolve to *different* genes as flagged bidirectional/adjacent pairs (not collapsed).

In [6]:
# (a) Confirm sequence-based twins map to adjacent loci
seq_tw = pd.read_parquet(f'{RESULTS}/CRISPRa_frameshift_twin_pairs.parquet')

def min_offset(a, b):
    best = None
    for c1, p1, s1 in aln.get(a, []):
        for c2, p2, s2 in aln.get(b, []):
            if c1 == c2 and s1 == s2:
                d = abs(p1 - p2)
                best = d if best is None else min(best, d)
    return best

seq_tw['genomic_min_offset'] = [min_offset(a, b) for a, b in zip(seq_tw.guide_a, seq_tw.guide_b)]
conf = (seq_tw['genomic_min_offset'] == 1).mean()
print(f'(a) sequence twins confirmed adjacent (offset==1): {conf:.1%} of {len(seq_tw):,}')
print('    offset distribution:', seq_tw['genomic_min_offset'].value_counts(dropna=False).to_dict())

# (b) Alignment-based discovery: offset-1 pairs among guides with <=10 loci (repeats excluded)
MULTI_CAP = 10
loc_bins = defaultdict(list)   # (chrom,strand) -> [(pos, guide)]
excluded = 0
for g, hits in aln.items():
    if len(hits) > MULTI_CAP:
        excluded += 1
        continue
    for c, p, s in hits:
        loc_bins[(c, s)].append((p, g))

seq_set = {frozenset((a, b)) for a, b in zip(seq_tw.guide_a, seq_tw.guide_b)}
aln_pairs = set()
for key, lst in loc_bins.items():
    lst.sort()
    for i in range(len(lst)):
        pi, gi = lst[i]
        j = i + 1
        while j < len(lst) and lst[j][0] - pi <= 1:
            pj, gj = lst[j]
            if gi != gj and (pj - pi) == 1:
                aln_pairs.add(frozenset((gi, gj)))
            j += 1

new_pairs = aln_pairs - seq_set
fg = dict(zip(annot.guide_id, annot.final_gene_id))
sym = dict(zip(master.guide_id, master.target_gene_symbol))
new_df = pd.DataFrame([dict(guide_a=min(p), guide_b=max(p),
                            sym_a=sym.get(min(p)), sym_b=sym.get(max(p)),
                            same_final_gene=(fg.get(min(p)) == fg.get(max(p)) and fg.get(min(p)) is not None
                                             and fg.get(min(p)) == fg.get(min(p))))
                       for p in new_pairs])
print(f'\n(b) alignment offset-1 pairs (<= {MULTI_CAP} loci): {len(aln_pairs):,}  '
      f'({excluded} repeat guides excluded)')
print(f'    NEW pairs missed by sequence method: {len(new_df):,}  '
      f'(same final gene: {int(new_df.same_final_gene.sum())}, different: {int((~new_df.same_final_gene).sum())})')
new_df.sort_values('same_final_gene', ascending=False).head(12)

(a) sequence twins confirmed adjacent (offset==1): 100.0% of 6,756
    offset distribution: {1: 6756}



(b) alignment offset-1 pairs (<= 10 loci): 6,834  (222 repeat guides excluded)
    NEW pairs missed by sequence method: 86  (same final gene: 26, different: 60)


,guide_a,guide_b,sym_a,sym_b,same_final_gene
43,PYCR3-3,PYCRL-2,PYCR3,PYCRL,True
29,C11orf83-4,UQCC3-2,C11orf83,UQCC3,True
73,TMEM35-1,TMEM35A-3,TMEM35,TMEM35A,True
70,C19orf84-3,LOC147646-2,C19orf84,LOC147646,True
69,RWDD3-4,TMEM56-RWDD3-1,RWDD3,TMEM56-RWDD3,True
24,C11orf97-4,LOC643037-1,C11orf97,LOC643037,True
68,ADAM15-2,ADAM15-4,ADAM15,ADAM15,True
28,SMAD9-2,SMAD9-3,SMAD9,SMAD9,True
30,LOC100505478-2,TEX48-3,LOC100505478,TEX48,True
17,CEP295-2,KIAA1731-3,CEP295,KIAA1731,True


In [7]:
# (c) Finalize twin edges = sequence twins + alignment-discovered pairs that share a final gene
new_same = new_df[new_df['same_final_gene']][['guide_a', 'guide_b']]
final_edges = pd.concat([seq_tw[['guide_a', 'guide_b']], new_same], ignore_index=True).drop_duplicates()

rank = master.set_index('guide_id')['guide_rank']
final_groups = collapse_twin_groups(final_edges, rank)
# twin_partner: the OTHER member(s) of a guide's twin group -- filled for EVERY twin (representative and
# collapsed alike), unlike twin_collapsed_into which only points from a dropped guide to its representative.
_members = final_groups.groupby('twin_group_id')['guide_id'].apply(list).to_dict()
final_groups['twin_partner'] = [
    ';'.join(sorted(g for g in _members[grp] if g != gid))
    for gid, grp in zip(final_groups['guide_id'], final_groups['twin_group_id'])]
print(f'final twin edges   : {len(final_edges):,}  '
      f'(sequence {len(seq_tw):,} + alignment-recovered {len(new_same):,})')
print(f'final twin groups  : {final_groups.twin_group_id.nunique():,}  '
      f'collapsing {int((~final_groups.is_representative).sum()):,} guides')

# apply to the annotated table
annot = annot.merge(final_groups.rename(columns={'is_representative': 'twin_representative',
                                                 'collapsed_into': 'twin_collapsed_into'}),
                    on='guide_id', how='left')
annot['is_twin'] = annot['twin_group_id'].notna()
annot['twin_representative'] = (~annot['is_twin']) | (annot['twin_representative'] == True)
annot['keep'] = annot['twin_representative'] & ~annot['flag_unaligned']

dedup_final = annot[annot['keep']].reset_index(drop=True)
print(f'\nfinal de-duplicated library: {len(dedup_final):,} guides '
      f'(Phase 2 sequence-only was 70,656)')
print(f'  guides dropped as twins   : {int((~annot.twin_representative).sum()):,}')
print(f'  guides dropped unaligned  : {int(annot.flag_unaligned.sum()):,}')

# example: A2M-2 (representative) now has a twin_partner even though twin_collapsed_into is empty
print('\ntwin_partner example:')
print(annot[annot.guide_id.isin(['A2M-2', 'A2M-3', 'A3GALT2-2', 'A3GALT2-4'])]
      [['guide_id', 'is_twin', 'twin_representative', 'twin_collapsed_into', 'twin_partner']].to_string(index=False))

# offset-1 pairs whose two guides resolve to DIFFERENT genes (bidirectional/adjacent, NOT collapsed)
adjacent_diff_gene = new_df[~new_df['same_final_gene']].reset_index(drop=True)
print(f'\nadjacent (offset-1) pairs across DIFFERENT genes, flagged not collapsed: {len(adjacent_diff_gene):,}')
adjacent_diff_gene.head(8)

final twin edges   : 6,782  (sequence 6,756 + alignment-recovered 26)
final twin groups  : 6,100  collapsing 6,782 guides



final de-duplicated library: 70,628 guides (Phase 2 sequence-only was 70,656)
  guides dropped as twins   : 6,782
  guides dropped unaligned  : 2

twin_partner example:
 guide_id  is_twin  twin_representative twin_collapsed_into twin_partner
    A2M-2     True                 True                 NaN        A2M-3
A3GALT2-2     True                 True                 NaN    A3GALT2-4
    A2M-3     True                False               A2M-2        A2M-2
A3GALT2-4     True                False           A3GALT2-2    A3GALT2-2

adjacent (offset-1) pairs across DIFFERENT genes, flagged not collapsed: 60


,guide_a,guide_b,sym_a,sym_b,same_final_gene
0,DAZ1-1,DAZ2-1,DAZ1,DAZ2,False
1,PTRH2-3,VMP1-3,PTRH2,VMP1,False
2,COQ6-4,FAM161B-3,COQ6,FAM161B,False
3,CFHR1-1,CFHR2-1,CFHR1,CFHR2,False
4,ARSK-1,TTC37-2,ARSK,TTC37,False
5,MBLAC2-4,POLR3G-1,MBLAC2,POLR3G,False
6,DTWD1-1,FAM227B-3,DTWD1,FAM227B,False
7,ERP44-2,INVS-3,ERP44,INVS,False


### Investigate the unaligned guides (why no perfect hg38 match?)

Re-align the guides that had 0 perfect matches with **one mismatch allowed** (`-N 1 -k 1`), then diff the
best hit against the genome and locate the nearest gene. Two distinct causes emerge:

- **on-target, single-base vs hg38** — best hit is at the *designed* gene's promoter with 1 mismatch
  (a SNP or a guide designed on an older assembly). Still a valid guide for that gene.
- **does not target the designed gene** — best imperfect hit is at a *different* gene → mis-designed /
  mislabeled guide.

In [8]:
from Bio.Seq import Seq

unaligned = master[master['n_loci'] == 0]
notes = []
if len(unaligned):
    ufa = f'{RESULTS}/CRISPRa_unaligned_guides.fa'
    with open(ufa, 'w') as f:
        for gid, proto in zip(unaligned['guide_id'], unaligned['protospacer']):
            f.write(f'>{gid}\n{proto[1:]}\n')   # 19 bp
    # best hit allowing 1 mismatch
    res = subprocess.run([BOWTIE2, '-x', BT2_INDEX, '-f', ufa, '--end-to-end',
                          '-L', '10', '-N', '1', '--score-min', 'L,-1.2,-0.9', '-k', '1'],
                         capture_output=True, text=True)
    best = {}
    for line in res.stdout.splitlines():
        if line.startswith('@'):
            continue
        p = line.split('\t')
        gid, flag, chrom, pos = p[0], int(p[1]), p[2], int(p[3]) - 1
        nm = next((int(t.split(':')[-1]) for t in p[11:] if t.startswith('NM:i:')), np.nan)
        best[gid] = None if (flag & 4) else (chrom, pos, '-' if flag & 16 else '+', p[9], nm)

    dgene = dict(zip(master['guide_id'], zip(master['gene_id'], master['current_symbol'])))
    for gid in unaligned['guide_id']:
        b = best.get(gid)
        d_gid, d_sym = dgene[gid]
        if b is None:
            notes.append(dict(guide_id=gid, designed_symbol=d_sym, best_hit=np.nan, n_mismatch=np.nan,
                              mismatch=np.nan, hit_gene=np.nan, hit_gene_dist=np.nan,
                              reason='no hit even with 1 mismatch', decision='exclude'))
            continue
        chrom, pos, strand, read, nm = b
        ref = genome[chrom][pos:pos + len(read)].seq.upper()
        mm = [(i, ref[i], read[i]) for i in range(len(read)) if read[i] != ref[i]]
        mm_str = ';'.join(f'pos{i}:{r}>{a}' for i, r, a in mm)
        hg_id, hg_name, hg_dist = nearest_tss(tss_index, chrom, pos)
        on_target = (hg_id == d_gid)
        if on_target:
            reason = (f'valid {d_sym} promoter guide ({int(hg_dist)}bp from TSS) with a single-base '
                      f'difference vs hg38 ({mm_str}) — SNP or older-assembly design')
            decision = 'keep'
        else:
            reason = (f'does NOT match designed {d_sym}; best imperfect hit ({nm} mm) is at '
                      f'{hg_name} promoter ({int(hg_dist)}bp from TSS on {chrom})')
            decision = 'exclude'
        notes.append(dict(guide_id=gid, designed_symbol=d_sym,
                          best_hit=f'{chrom}:{pos}({strand})', n_mismatch=nm, mismatch=mm_str,
                          hit_gene=hg_name, hit_gene_dist=hg_dist, reason=reason, decision=decision))

unaligned_notes = pd.DataFrame(notes)

# --- Apply verdicts ---
# 'keep' = on-target guide with a single-base mismatch vs hg38: rescue it back into the library,
#          reassign to its designed gene (note records the mismatch). 'exclude' = mislabeled/off-target.
if len(unaligned_notes):
    rescue_ids = set(unaligned_notes.loc[unaligned_notes['decision'] == 'keep', 'guide_id'])
    for gid in rescue_ids:
        m = annot['guide_id'] == gid
        annot.loc[m, 'final_gene_id'] = annot.loc[m, 'gene_id']
        annot.loc[m, 'final_gene_name'] = annot.loc[m, 'current_symbol']
        annot.loc[m, 'assignment'] = 'unaligned_rescued'
        annot.loc[m, 'final_gene_source'] = 'unaligned_rescued'
        annot.loc[m, 'on_target'] = True
        annot.loc[m, 'keep'] = annot.loc[m, 'twin_representative']   # keep unless a collapsed twin
    dedup_final = annot[annot['keep']].reset_index(drop=True)        # recompute library after rescue
    print(f"rescued {len(rescue_ids)} unaligned guide(s) into the library; "
          f"final de-duplicated library now {len(dedup_final):,}")

for _, r in unaligned_notes.iterrows():
    print(f"{r.guide_id} (designed {r.designed_symbol}) [{r.decision}]: {r.reason}")
unaligned_notes

rescued 1 unaligned guide(s) into the library; final de-duplicated library now 70,629
DGAT1-3 (designed DGAT1) [keep]: valid DGAT1 promoter guide (94bp from TSS) with a single-base difference vs hg38 (pos12:G>A) — SNP or older-assembly design
GPR3-4 (designed GPR3) [exclude]: does NOT match designed GPR3; best imperfect hit (1 mm) is at RALGPS1 promoter (1bp from TSS on chr9)


,guide_id,designed_symbol,best_hit,n_mismatch,mismatch,hit_gene,hit_gene_dist,reason,decision
0,DGAT1-3,DGAT1,chr8:144327004(+),1,pos12:G>A,DGAT1,94,valid DGAT1 promoter guide (94bp from TSS) wit...,keep
1,GPR3-4,GPR3,chr9:126914828(-),1,pos5:C>A,RALGPS1,1,does NOT match designed GPR3; best imperfect h...,exclude


## Phase 6 — neighborhood & off-target annotation

Adds the reference `sgrna_df_final` neighbor columns for every targeting guide, computed from the
sorted-TSS index (fast) rather than per-row scans over the whole gene table:

- `nearby_gene_within_{2,10,20,30}kb` — gene_ids whose TSS is within each tier of the guide's locus.
- `nearest_within2kb_gene_*` and `nearest_within2kb_nontarget_gene_*` — nearest gene / nearest **non-target**
  gene within 2 kb (co-targeting risk).
- `putative_bidirectional_promoter` — a gene on each side pointing away (divergent) within 2 kb.
- `other_alignment_chromosome` / `other_alignment_pos` — the secondary loci of multi-mapping guides.
- `nearest_nontarget_gene_*` — nearest non-target gene across **all** of a guide's alignment positions.

In [9]:
# gene metadata for orientation / names
gm = genes.drop_duplicates('gene_id').set_index('gene_id')
gene_name = gm['gene_name'].to_dict()
gene_strand = gm['strand'].to_dict()
gene_start = gm['gene_start'].to_dict()
gene_end = gm['gene_end'].to_dict()

OTHER_CAP = 10   # cap the listed secondary loci (repeat guides can align to >100k sites; n_loci has the true count)


def nearest_nontarget_all_loci(guide, target_id, target_name, window=500000):
    """Nearest non-target gene across all loci (<= window). A 'non-target' must be a genuinely
    DIFFERENT gene -- exclude the target id and any same-named copy (PAR X/Y, alt-haplotype)."""
    best = None
    for ch, p, s in aln.get(guide, []):
        for g, d in genes_near(tss_index, ch, p, window).items():
            if g != target_id and gene_name.get(g) != target_name and (best is None or d < best[2]):
                best = (g, gene_name.get(g), d)
    return best if best else (np.nan, np.nan, np.nan)


def phase6_row(r):
    guide, chrom, pos = r['guide_id'], r['chrom'], r['pos']
    target, target_name = r['final_gene_id'], r['final_gene_name']
    out = dict(nearby_gene_within_2kb=[], nearby_gene_within_10kb=[],
               nearest_within2kb_gene_id=np.nan, nearest_within2kb_gene_name=np.nan,
               nearest_within2kb_gene_dist=np.nan,
               nearest_within2kb_nontarget_gene_id=np.nan, nearest_within2kb_nontarget_gene_name=np.nan,
               nearest_within2kb_nontarget_gene_dist=np.nan,
               putative_bidirectional_promoter=False,
               other_alignment_chromosome=[], other_alignment_pos=[],
               nearest_nontarget_gene_id=np.nan, nearest_nontarget_gene_name=np.nan,
               nearest_nontarget_gene_dist=np.nan)
    # secondary alignment loci (multi-mappers), capped at OTHER_CAP examples (full count is n_loci)
    loci = aln.get(guide, [])
    others = sorted((c, p) for (c, p, s) in loci if not (c == chrom and p == pos))
    out['other_alignment_chromosome'] = [c for c, p in others[:OTHER_CAP]]
    out['other_alignment_pos'] = [p for c, p in others[:OTHER_CAP]]

    if pd.isna(chrom) or pd.isna(pos):
        return pd.Series(out)
    pos = int(pos)
    near = genes_near(tss_index, chrom, pos, 10000)
    within2 = [g for g, d in near.items() if d <= 2000]
    within10 = list(near.keys())
    out['nearby_gene_within_2kb'] = sorted(within2, key=lambda g: near[g])
    out['nearby_gene_within_10kb'] = sorted(within10, key=lambda g: near[g])

    w2 = {g: near[g] for g in within2}
    if w2:
        ng = min(w2, key=w2.get)
        out.update(nearest_within2kb_gene_id=ng, nearest_within2kb_gene_name=gene_name.get(ng),
                   nearest_within2kb_gene_dist=w2[ng])
        # non-target = different gene id AND different gene name (excludes PAR/alt same-gene copies)
        nt = {g: d for g, d in w2.items() if g != target and gene_name.get(g) != target_name}
        if nt:
            ntg = min(nt, key=nt.get)
            out.update(nearest_within2kb_nontarget_gene_id=ntg,
                       nearest_within2kb_nontarget_gene_name=gene_name.get(ntg),
                       nearest_within2kb_nontarget_gene_dist=nt[ntg])
        # bidirectional promoter: a gene on each side pointing away
        left = any(gene_strand.get(g) == '-' and gene_start.get(g, np.inf) < pos for g in w2)
        right = any(gene_strand.get(g) == '+' and gene_end.get(g, -np.inf) > pos for g in w2)
        out['putative_bidirectional_promoter'] = bool(left and right)

    nn = nearest_nontarget_all_loci(guide, target, target_name)
    out.update(nearest_nontarget_gene_id=nn[0], nearest_nontarget_gene_name=nn[1],
               nearest_nontarget_gene_dist=nn[2])
    return pd.Series(out)


phase6 = annot.apply(phase6_row, axis=1)
annot = pd.concat([annot, phase6], axis=1)
print('Phase 6 columns added:', list(phase6.columns))
print(f"other_alignment lists capped at {OTHER_CAP} (n_loci holds the true count)")
print(f"\nputative bidirectional promoters : {int(annot.putative_bidirectional_promoter.sum()):,}")
print(f"guides with a non-target gene within 2kb: {int(annot.nearest_within2kb_nontarget_gene_id.notna().sum()):,}")
annot.loc[annot.putative_bidirectional_promoter,
          ['guide_id', 'final_gene_name', 'nearest_within2kb_nontarget_gene_name',
           'nearest_within2kb_nontarget_gene_dist']].head(6)

Phase 6 columns added: ['nearby_gene_within_2kb', 'nearby_gene_within_10kb', 'nearest_within2kb_gene_id', 'nearest_within2kb_gene_name', 'nearest_within2kb_gene_dist', 'nearest_within2kb_nontarget_gene_id', 'nearest_within2kb_nontarget_gene_name', 'nearest_within2kb_nontarget_gene_dist', 'putative_bidirectional_promoter', 'other_alignment_chromosome', 'other_alignment_pos', 'nearest_nontarget_gene_id', 'nearest_nontarget_gene_name', 'nearest_nontarget_gene_dist']
other_alignment lists capped at 10 (n_loci holds the true count)

putative bidirectional promoters : 11,837
guides with a non-target gene within 2kb: 14,526


,guide_id,final_gene_name,nearest_within2kb_nontarget_gene_name,nearest_within2kb_nontarget_gene_dist
30,AAGAB-1,AAGAB,IQCH,65.0
31,AAGAB-2,AAGAB,IQCH,31.0
32,AAK1-1,AAK1,ANXA4,640.0
33,AAK1-2,AAK1,ANXA4,654.0
34,AAMDC-1,AAMDC,RSF1,115.0
35,AAMDC-2,AAMDC,RSF1,114.0


## Step A — complete the annotated table: all sgRNA (incl. NO-TARGET) + unified `note`

Align the 958 NO-TARGET (non-targeting control) guides the same way, so any NTC that unexpectedly maps to
a gene promoter is flagged. Then build the final annotated table over **all 78,370 guides** with a single
human-readable `note` column (folding in the assignment, twin, PAM, and unaligned-guide reasons) and an
`is_targeting` flag.

In [10]:
# --- Align the NO-TARGET (non-targeting control) guides ---
ntc = pd.read_parquet(f'{RESULTS}/CRISPRa_NO-TARGET_sgRNA.parquet')
ntc_fa = f'{RESULTS}/CRISPRa_NO-TARGET_sgRNA.fa'
with open(ntc_fa, 'w') as f:
    for gid, proto in zip(ntc['guide_id'], ntc['protospacer']):
        f.write(f'>{gid}\n{proto[1:]}\n')

ntc_sam = f'{RESULTS}/CRISPRa_NO-TARGET_alignment.sam'
if not os.path.exists(ntc_sam):
    subprocess.run([BOWTIE2, '-x', BT2_INDEX, '-f', ntc_fa, '-S', ntc_sam,
                    '--end-to-end', '-L', '19', '-N', '0', '--score-min', 'C,0,0', '-a', '-p', '8'],
                   capture_output=True, text=True)

ntc_aln = defaultdict(list)
with pysam.AlignmentFile(ntc_sam, 'r') as sam:
    for r in sam:
        if not r.is_unmapped:
            ntc_aln[r.query_name].append((r.reference_name, int(r.reference_start),
                                          '-' if r.is_reverse else '+'))

# nearest-gene primary locus per NTC (they have no designed gene)
rows = []
for gid in ntc['guide_id']:
    hits = ntc_aln.get(gid, [])
    if not hits:
        rows.append(dict(guide_id=gid, n_loci=0, chrom=np.nan, pos=np.nan, strand=np.nan,
                         has_pam=np.nan, locus_gene_id=np.nan, locus_gene_name=np.nan, dist_locus_tss=np.nan))
        continue
    best = None
    for ch, p, s in hits:
        lg_id, lg_name, lg_d = nearest_tss(tss_index, ch, p)
        if best is None or _d(lg_d) < _d(best[5]):
            best = (ch, p, s, lg_id, lg_name, lg_d)
    ch, p, s, lg_id, lg_name, lg_d = best
    rows.append(dict(guide_id=gid, n_loci=len(hits), chrom=ch, pos=p, strand=s,
                     has_pam=has_ngg_pam(genome, ch, p, s),
                     locus_gene_id=lg_id, locus_gene_name=lg_name, dist_locus_tss=lg_d))
ntc_full = ntc.merge(pd.DataFrame(rows), on='guide_id')
ntc_full['flag_ntc_off_target'] = ntc_full['dist_locus_tss'] <= TSS_WINDOW

print(f'NTC guides             : {len(ntc_full):,}')
print(f'  aligned (>=1 locus)  : {int((ntc_full.n_loci>0).sum()):,}')
print(f'  within {TSS_WINDOW}bp of a gene TSS (off-target concern): {int(ntc_full.flag_ntc_off_target.sum()):,}')
ntc_full[ntc_full.flag_ntc_off_target][['guide_id','chrom','pos','locus_gene_name','dist_locus_tss']].head()

NTC guides             : 958
  aligned (>=1 locus)  : 10
  within 2000bp of a gene TSS (off-target concern): 0


,guide_id,chrom,pos,locus_gene_name,dist_locus_tss


In [11]:
def _fmt(x):
    return '' if pd.isna(x) else str(int(x))

un_reason = dict(zip(unaligned_notes['guide_id'], unaligned_notes['reason'])) if len(unaligned_notes) else {}


def note_targeting(r):
    """Single source of truth for a targeting guide's note.

    The note carries ONLY the reason a guide is not a clean on-target guide.
    Clean, kept, single-locus (or PAR) on-target guides get an empty note; their
    status is fully captured by the structured columns. Descriptive neighborhood
    info (bidirectional promoter, nearby non-target gene) lives only in its own
    columns, never in the note.
    """
    if not r['flag']:                     # clean guide -> no note
        return ''
    a, fn = r['assignment'], r['final_gene_name']
    # base = why this guide's TARGET is not a straightforward on-target call
    # (suppressed for on-target / distal / PAR-matching and for frameshift twins,
    #  whose target equals their representative's).
    if a == 'unaligned':
        base = un_reason.get(r['guide_id'], 'no perfect hg38 match') + ' [EXCLUDED]'
    elif a == 'unaligned_rescued':
        base = un_reason.get(r['guide_id'], 'no perfect hg38 match') + ' [KEPT: reassigned to designed gene]'
    elif a == 'locus_reassigned':
        base = f"TARGET MISMATCH: designed {r['designed_gene_name']} but aligns to {fn} ({_fmt(r['dist_locus_tss'])}bp from its TSS)"
    elif a == 'par_gene' and isinstance(r['designed_gene_name'], str) and fn != r['designed_gene_name']:
        base = f"PAR gene {fn} (chrX/Y) but designed for {r['designed_gene_name']} — TARGET MISMATCH"
    elif a == 'locus_assigned':
        base = f"designed name unresolved; locus-assigned {fn} ({_fmt(r['dist_locus_tss'])}bp from TSS)"
    elif a == 'retired_identifier':
        base = f"retired/withdrawn identifier '{r['designed_symbol']}'; alt contig {r['chrom']}, no current Ensembl gene (label kept, id empty)"
    else:
        base = ''
    if r['is_twin'] and r['on_target']:
        base = ''
    # appended issues
    extra = []
    if r['flag_no_pam'] is True:
        extra.append('no NGG PAM')
    if r['is_twin']:
        extra.append(f"frameshift-twin of {r['twin_partner']} (collapsed, kept)" if not r['twin_representative']
                     else f"frameshift-twin of {r['twin_partner']} (kept)")
    if r['n_loci'] and r['n_loci'] > 1 and not r['flag_par']:
        extra.append(f"{int(r['n_loci'])} genomic loci (multi-mapping)")
    return '; '.join(([base] if base else []) + extra)


def note_ntc(r):
    if r['n_loci'] == 0:
        return 'non-targeting control (no perfect genomic match)'
    if r['flag_ntc_off_target']:
        return f"non-targeting control BUT aligns {_fmt(r['dist_locus_tss'])}bp from {r['locus_gene_name']} TSS — possible off-target"
    return f"non-targeting control (intergenic; nearest TSS {_fmt(r['dist_locus_tss'])}bp)"


PHASE6_COLS = ['nearby_gene_within_2kb', 'nearby_gene_within_10kb',
               'nearest_within2kb_gene_id', 'nearest_within2kb_gene_name', 'nearest_within2kb_gene_dist',
               'nearest_within2kb_nontarget_gene_id', 'nearest_within2kb_nontarget_gene_name',
               'nearest_within2kb_nontarget_gene_dist', 'putative_bidirectional_promoter',
               'other_alignment_chromosome', 'other_alignment_pos',
               'nearest_nontarget_gene_id', 'nearest_nontarget_gene_name', 'nearest_nontarget_gene_dist']

OUT_COLS = ['guide_id', 'is_targeting', 'protospacer', 'Set', 'guide_rank', 'gc_fraction', 'pool',
            'designed_symbol', 'designed_gene_id', 'designed_gene_name', 'name_resolution',
            'n_loci', 'chrom', 'pos', 'strand', 'has_pam',
            'locus_gene_id', 'locus_gene_name', 'dist_locus_tss', 'dist_designed_tss',
            'assignment', 'final_gene_id', 'final_gene_name', 'final_gene_source', 'on_target',
            'is_twin', 'twin_partner', 'twin_group_id', 'twin_representative', 'twin_collapsed_into', 'keep',
            'flag_unaligned', 'flag_no_pam', 'flag_far_from_tss', 'flag_target_mismatch',
            'flag_designed_distal', 'flag_name_rescued_by_locus', 'flag_retired_identifier',
            'flag_par', 'flag_ntc_off_target'] + PHASE6_COLS + ['flag', 'note']

# --- targeting guides: compute the consolidated review flag, then the note (one place) ---
tgt = annot.rename(columns={'target_gene_symbol': 'designed_symbol', 'gene_id': 'designed_gene_id',
                            'current_symbol': 'designed_gene_name'}).copy()
tgt['is_targeting'] = True
tgt['flag_ntc_off_target'] = False
# flag = NOT a clean, kept, single-locus (or PAR) on-target guide with a PAM
tgt['flag'] = ~(tgt['on_target'] & (tgt['has_pam'] == True)
                & ((tgt['n_loci'] == 1) | tgt['flag_par']) & tgt['keep'])
tgt['note'] = tgt.apply(note_targeting, axis=1)

ntc_out = ntc_full.rename(columns={'target_gene_symbol': 'designed_symbol'}).copy()
ntc_out['is_targeting'] = False
ntc_out['assignment'] = 'non_targeting'
ntc_out['final_gene_source'] = 'non_targeting'
ntc_out['flag'] = ntc_out['flag_ntc_off_target']
ntc_out['note'] = ntc_out.apply(note_ntc, axis=1)

annotated_all = pd.concat([tgt.reindex(columns=OUT_COLS), ntc_out.reindex(columns=OUT_COLS)],
                          ignore_index=True)
# fill NTC-side gaps and force bool dtype (concat with NTC NaNs can upcast bools to object/float)
flag_cols = [c for c in OUT_COLS if c.startswith('flag_')] + ['on_target', 'putative_bidirectional_promoter', 'flag']
annotated_all[flag_cols] = annotated_all[flag_cols].fillna(False)
annotated_all['is_twin'] = annotated_all['is_twin'].fillna(False)
annotated_all['twin_representative'] = annotated_all['twin_representative'].fillna(True)
annotated_all['twin_partner'] = annotated_all['twin_partner'].fillna('')   # non-twins: no partner
annotated_all['keep'] = annotated_all['keep'].fillna(True)   # NTCs are kept controls
bool_cols = flag_cols + ['is_twin', 'twin_representative', 'keep']
annotated_all[bool_cols] = annotated_all[bool_cols].astype(bool)
# list columns: NTC rows have NaN -> empty list
for c in ['nearby_gene_within_2kb', 'nearby_gene_within_10kb', 'other_alignment_chromosome', 'other_alignment_pos']:
    annotated_all[c] = annotated_all[c].apply(lambda x: x if isinstance(x, list) else [])

n_note = int((annotated_all['note'].fillna('').astype(str).str.len() > 0).sum())
print(f'annotated_all rows: {len(annotated_all):,}  cols: {annotated_all.shape[1]}  '
      f'(targeting {int(annotated_all.is_targeting.sum()):,} + NTC {int((~annotated_all.is_targeting).sum()):,})')
print(f'kept (library) : {int(annotated_all.keep.sum()):,}   flagged targeting: '
      f'{int(annotated_all[annotated_all.is_targeting].flag.sum()):,}   rows with a note: {n_note:,}')
print('\nexample notes:')
for gid in ['A2M-2', 'A2M-3', 'DDB1-1', 'ADORA3-1', 'GSTT1-1']:
    row = annotated_all[annotated_all.guide_id == gid]
    if len(row):
        print(f"  {gid:12} twin_partner={row.iloc[0]['twin_partner'] or '-':10} | {row.iloc[0]['note']}")

annotated_all rows: 78,370  cols: 56  (targeting 77,412 + NTC 958)
kept (library) : 71,587   flagged targeting: 13,516   rows with a note: 14,474

example notes:
  A2M-2        twin_partner=A2M-3      | 
  A2M-3        twin_partner=A2M-2      | frameshift-twin of A2M-2 (collapsed, kept); 2 genomic loci (multi-mapping)
  DDB1-1       twin_partner=DDB1-3     | 
  ADORA3-1     twin_partner=-          | TARGET MISMATCH: designed ADORA3 but aligns to TMIGD3 (87bp from its TSS)
  GSTT1-1      twin_partner=-          | 


### Save Phase 3-5 outputs

In [12]:
def save(df, stem):
    df.to_parquet(f'{RESULTS}/{stem}.parquet')
    df.to_csv(f'{RESULTS}/{stem}.csv', index=False)
    print(f'  {stem:<48} {len(df):>7,} rows')

# compact gene-correction table (targeting guides only)
corrections = annot[['guide_id', 'target_gene_symbol', 'current_symbol', 'gene_id',
                     'name_resolution', 'n_loci', 'chrom', 'pos', 'strand', 'has_pam',
                     'dist_designed_tss', 'dist_locus_tss', 'assignment', 'on_target',
                     'final_gene_id', 'final_gene_name', 'final_gene_source',
                     'flag_unaligned', 'flag_no_pam', 'flag_target_mismatch',
                     'flag_designed_distal', 'flag_name_rescued_by_locus',
                     'flag_retired_identifier', 'flag_par']].copy()

print('Writing to results/ ...')
save(annotated_all,       'CRISPRa_targeting_sgRNA_annotated')        # ALL sgRNA (targeting + NTC) + note column
save(dedup_final,         'CRISPRa_targeting_sgRNA_final_deduplicated')
save(corrections,         'CRISPRa_gene_assignment_corrections')
save(seq_tw,              'CRISPRa_frameshift_twin_pairs_confirmed')  # + genomic_min_offset
save(new_df,              'CRISPRa_frameshift_twins_alignment_recovered')
save(final_groups,        'CRISPRa_frameshift_twin_groups_final')
save(adjacent_diff_gene,  'CRISPRa_adjacent_offset1_different_gene')
save(unaligned_notes,     'CRISPRa_unaligned_guides_notes')          # per-guide reason + decision
print('Done.')

Writing to results/ ...


  CRISPRa_targeting_sgRNA_annotated                 78,370 rows


  CRISPRa_targeting_sgRNA_final_deduplicated        70,629 rows


  CRISPRa_gene_assignment_corrections               77,412 rows
  CRISPRa_frameshift_twin_pairs_confirmed            6,756 rows
  CRISPRa_frameshift_twins_alignment_recovered          86 rows
  CRISPRa_frameshift_twin_groups_final              12,882 rows
  CRISPRa_adjacent_offset1_different_gene               60 rows
  CRISPRa_unaligned_guides_notes                         2 rows
Done.


## Summary

**Phase 3-5 outputs** (`results/`):
- `CRISPRa_targeting_sgRNA_annotated` — every guide with alignment locus, PAM, locus-based gene, final
  gene call, all flags, and final twin group. The master annotation.
- `CRISPRa_targeting_sgRNA_final_deduplicated` — collapsed library (twins + unaligned removed).
- `CRISPRa_gene_assignment_corrections` — compact per-guide correction table.
- `CRISPRa_frameshift_twin_pairs_confirmed` — Phase 2 twins + `genomic_min_offset` (all == 1).
- `CRISPRa_frameshift_twins_alignment_recovered` — twins the 19 bp alignment found that the 20 bp
  sequence method missed.
- `CRISPRa_frameshift_twin_groups_final` / `CRISPRa_adjacent_offset1_different_gene` — final groups; and
  offset-1 pairs spanning two different genes (bidirectional/adjacent), kept not collapsed.

**Twin revisit result:** 100% of sequence twins confirmed genomically adjacent; alignment recovered a
handful more same-gene twins (5' base-dependent). The frameshift-twin call is now locus-validated.

Guides flagged `unaligned` / `target_mismatch` / `no_pam` / `designed_distal` are the data-cleanup
candidates (outdated design, wrong-gene, or distal-from-TSS).